# Segment DP v2: Joint FC + Non-FC Optimization

모든 encoder 레이어(FC chunks + Non-FC conv/fc)를 **동시에** 최적화하는 Segment DP.

### v1 → v2 변경사항
| | v1 (이전) | v2 (수정) |
|---|---|---|
| 최적화 블록 | FC 32 chunks only | FC 32 + Non-FC = **40 blocks** |
| Unique operating points | ~15 | **~100+** |
| C_steps (budget 해상도) | 2000 | **3000** |
| Savings step | 0.25% | **0.1%** |
| 곡선 형태 | 이산적/계단 | **부드러운 Pareto** |

### 실행 순서
1. **Cell 1**: 환경 셋업
2. **Cell 2**: Segment DP v2 — CLNet + CRNet + MT-AE 전부 (~40분)
3. **Cell 3**: Figure 재생성

## Cell 1: Environment Setup

In [1]:
import os, sys

PROJECT_ROOT = '/content/drive/MyDrive/MambaCompression'
MAMBAIC_ROOT = os.path.join(PROJECT_ROOT, 'MambaIC')

if not os.path.isdir(PROJECT_ROOT):
    from google.colab import drive
    drive.mount('/content/drive')

os.chdir(MAMBAIC_ROOT)
if MAMBAIC_ROOT not in sys.path:
    sys.path.insert(0, MAMBAIC_ROOT)

setup_path = os.path.join(PROJECT_ROOT, 'setup_colab.py')
if os.path.isfile(setup_path):
    exec(open(setup_path).read())
else:
    !pip install -q einops scipy tqdm thop fvcore pybind11
!pip install -q pulp 2>/dev/null | tail -1

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Verify key files exist
baselines_root = os.path.join(os.path.dirname(MAMBAIC_ROOT), 'baselines')
for f in [
    os.path.join(baselines_root, 'CLNet-master/checkpoints/out4.pth'),
    os.path.join(baselines_root, 'CRNet-master/checkpoints/out_04.pth'),
    os.path.join(MAMBAIC_ROOT, 'results/csv/rpmpq_v2_perfect_rates.csv'),
]:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'  [{status}] {os.path.relpath(f, PROJECT_ROOT)}')

# Check existing omega caches
csv_dir = os.path.join(MAMBAIC_ROOT, 'results', 'csv')
for model in ['clnet', 'crnet']:
    old = os.path.join(csv_dir, f'segment_dp_omegas_{model}.csv')
    new = os.path.join(csv_dir, f'segment_dp_omegas_v2_{model}.csv')
    if os.path.exists(new):
        print(f'  [CACHED] v2 omegas for {model} (will skip GPU collection)')
    elif os.path.exists(old):
        print(f'  [REUSE] v1 FC omegas for {model} (only non-FC needs GPU)')
    else:
        print(f'  [NEW] {model}: full omega collection needed')

print('\nReady.')

Mounted at /content/drive
=== 1. Core Dependencies ===
[  0.0s] pip install core deps...
[  9.6s] core deps done

=== 2. VMamba CUDA Kernel (ss2d) ===
Current GPU: Tesla T4 (sm_75)
Cache arch matches current GPU (sm_75) ✓
Cache found! Restoring 1 kernel files...
[ 15.2s] copying .so from cache...
  Restored: selective_scan_cuda_oflex.cpython-312-x86_64-linux-gnu.so -> /usr/local/lib/python3.12/dist-packages/selective_scan_cuda_oflex.cpython-312-x86_64-linux-gnu.so
[ 17.4s] .so copy done
[ 17.4s] importing selective_scan_cuda_oflex...
[ 17.4s] selective_scan_cuda_oflex imported OK (sm_75)
selective_scan_cuda_oflex imported OK (sm_75)
[ 17.4s] setup_colab.py done

=== Setup Complete ===
Project: /content/drive/MyDrive/MambaCompression
CUDA: True
GPU: Tesla T4
  [OK] baselines/CLNet-master/checkpoints/out4.pth
  [OK] baselines/CRNet-master/checkpoints/out_04.pth
  [OK] MambaIC/results/csv/rpmpq_v2_perfect_rates.csv
  [CACHED] v2 omegas for clnet (will skip GPU collection)
  [CACHED] v2 om

## Cell 2: Run Segment DP v2 — ALL models

CsiNet (Keras 원본) + CLNet + CRNet + MT-AE 전부 한번에 실행.

- **CsiNet**: 원본 Keras 모델 직접 사용 (PyTorch 변환 X → NMSE -8.95 dB 정확히 재현)
- **CLNet, CRNet, MT-AE**: PyTorch 기반

**예상 시간: ~50분**

In [ ]:
import os, sys, time, glob

MAMBAIC_ROOT = '/content/drive/MyDrive/MambaCompression/MambaIC'
os.chdir(MAMBAIC_ROOT)
if MAMBAIC_ROOT not in sys.path:
    sys.path.insert(0, MAMBAIC_ROOT)

csv_dir = os.path.join(MAMBAIC_ROOT, 'results', 'csv')

# ── v2 omega cache는 유지 (이미 수집된 CLNet/CRNet 재활용) ──
# ── 결과 CSV만 삭제해서 전체 DP sweep 재실행 ──
old_result = os.path.join(csv_dir, 'segment_dp_baselines.csv')
if os.path.exists(old_result):
    os.remove(old_result)
    print('Deleted: segment_dp_baselines.csv (results only, omega caches kept)')

# 어떤 omega cache가 있는지 확인
for f in sorted(glob.glob(os.path.join(csv_dir, 'segment_dp_omegas_v2_*.csv'))):
    print(f'  [CACHED] {os.path.basename(f)}')

t0 = time.time()

# ── Segment DP v2: CsiNet(Keras) + CLNet + CRNet + MT-AE 전부 ──
!python analysis/segment_dp_baselines.py

elapsed = time.time() - t0
print(f'\nTotal time: {elapsed/60:.1f} min')

# ── Summary ──
import pandas as pd
csv_path = os.path.join(csv_dir, 'segment_dp_baselines.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print('\n=== All Models Joint DP v2 ===')
    for model in sorted(df['model'].unique()):
        sub = df[(df['model']==model) & (df['method']=='segment-dp')]
        if len(sub) == 0:
            continue
        n_unique = sub['segmentation'].nunique()
        n_sav = sub['actual_saving'].nunique()
        print(f'  {model}: {n_unique} unique policies, {n_sav} unique savings, '
              f'NMSE: {sub["nmse_db"].min():.2f} ~ {sub["nmse_db"].max():.2f} dB')

  [CACHED] segment_dp_omegas_v2_clnet.csv
  [CACHED] segment_dp_omegas_v2_crnet.csv
  [CACHED] segment_dp_omegas_v2_mt-ae.csv
  SEGMENT DP ON BASELINE MODELS
Device: CUDA
Test samples: 20000
2026-03-22 11:03:36.825273: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774177416.858401   10578 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774177416.865595   10578 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774177416.883798   10578 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774177416.883824   10578 computation_placer.cc:177] comput

## Cell 3: Regenerate Figures

업데이트된 segment_dp_baselines.csv로 Fig 1(b) 재생성.

In [ ]:
import os
MAMBAIC_ROOT = '/content/drive/MyDrive/MambaCompression/MambaIC'
os.chdir(MAMBAIC_ROOT)

# Regenerate all paper figures
print('Regenerating all paper figures...')
!python analysis/generate_paper_figures.py

print('\nDone. Check figures/ directory for updated plots.')

# Show summary
import pandas as pd
csv_path = os.path.join(MAMBAIC_ROOT, 'results', 'csv', 'segment_dp_baselines.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print('\n=== Segment DP v2 Results ===')
    for model in sorted(df['model'].unique()):
        sub = df[(df['model']==model) & (df['method']=='segment-dp')]
        n_pol = sub['segmentation'].nunique()
        n_sav = sub['actual_saving'].nunique()
        print(f'  {model}: {n_pol} policies, {n_sav} savings, '
              f'NMSE: {sub["nmse_db"].min():.2f} ~ {sub["nmse_db"].max():.2f} dB')

## Cell 4: (Optional) Visual Comparison — Old vs New

이전 per-layer ILP 결과와 새 segment-DP v2 결과를 비교.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import numpy as np
import pandas as pd
import os

MAMBAIC_ROOT = '/content/drive/MyDrive/MambaCompression/MambaIC'
RESULTS_CSV = os.path.join(MAMBAIC_ROOT, 'results', 'csv')
FIGURES_DIR = os.path.join(MAMBAIC_ROOT, 'results', 'plots')

STYLE = {
    'font.size': 13, 'axes.labelsize': 13, 'axes.titlesize': 13,
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 10,
    'lines.linewidth': 2.0, 'lines.markersize': 6, 'figure.dpi': 150,
}

def _monotone(y):
    y = np.array(y, dtype=float)
    best = float('-inf')
    for i in range(len(y)):
        if y[i] > best: best = y[i]
        else: y[i] = best
    return y

with plt.rc_context(STYLE):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for idx, model in enumerate(['CLNet', 'CRNet']):
        ax = axes[idx]

        # Old per-layer ILP
        old_csv = os.path.join(RESULTS_CSV,
            f'mp_policy_lut_{model.lower()}_cr4_out_a8.csv')
        if os.path.exists(old_csv):
            df_old = pd.read_csv(old_csv).sort_values('Actual_Saving')
            nmse_col = 'NMSE_dB' if 'NMSE_dB' in df_old.columns else 'NMSE_dB_ILP'
            x_old = df_old['Actual_Saving'].values
            y_old = _monotone(df_old[nmse_col].values)
            ax.plot(x_old, y_old, '--', color='#999999', linewidth=1.5,
                    label='v1: per-layer ILP', alpha=0.8)

        # New segment-DP v2
        seg_csv = os.path.join(RESULTS_CSV, 'segment_dp_baselines.csv')
        if os.path.exists(seg_csv):
            df_new = pd.read_csv(seg_csv)
            sub = df_new[(df_new['model']==model) &
                         (df_new['method']=='segment-dp')].sort_values('actual_saving')
            if len(sub) > 0:
                x_new = sub['actual_saving'].values
                y_new = _monotone(sub['nmse_db'].values)
                ax.plot(x_new, y_new, '-', color='#1f77b4', linewidth=2.0,
                        marker='o', markersize=4, markevery=3,
                        label='v2: joint segment-DP')

        ax.set_title(f'{model}')
        ax.set_xlabel('BOPs Saving vs. FP32 (%)')
        ax.set_ylabel('NMSE (dB)')
        ax.set_xlim(84, 97)
        ax.xaxis.set_major_locator(MultipleLocator(2))
        ax.legend(loc='upper left')
        ax.grid(True, linestyle='--', alpha=0.4, color='#cccccc')

    fig.suptitle('Segment DP v1 (FC-only) vs v2 (Joint)', fontsize=14, y=1.02)
    fig.tight_layout()
    out_path = os.path.join(FIGURES_DIR, 'segment_dp_v1_vs_v2.png')
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    print(f'Saved: {out_path}')
    plt.show()

## Cell 5: Regenerate ALL stale CSVs (v2, 0.1% step)

아래 순서로 stale CSV 전부 재수집:

| Step | Script | Output | GPU | 예상시간 |
|------|--------|--------|-----|---------|
| 1 | `budget_allocation.py` | `budget_allocation.csv` | No | ~2분 |
| 2 | `eval_full_comparison.py` | `full_comparison.csv` | **Yes** | ~60분 |
| 3 | `eval_outage_multi_snr_sweep.py` | `outage_multi_snr_sweep.csv` | **Yes** | ~30분 |
| 4 | `budget_allocation_outage.py` | `outage_curves_per_bin.csv`, `budget_allocation_outage.csv`, `complete_eval_mtae.csv` | **Yes** | ~20분 |
| 5 | `generate_paper_figures.py` | `figures/*.pdf` | No | ~1분 |

**총 예상: ~2시간** (전부 0.1% step = 121 saving levels)

In [ ]:
import os, sys, time
from datetime import datetime

MAMBAIC_ROOT = '/content/drive/MyDrive/MambaCompression/MambaIC'
os.chdir(MAMBAIC_ROOT)
if MAMBAIC_ROOT not in sys.path:
    sys.path.insert(0, MAMBAIC_ROOT)

csv_dir = os.path.join(MAMBAIC_ROOT, 'results', 'csv')

# ── Verify v2 prerequisites ──
print('=== Prerequisites ===')
for f in ['segment_dp_baselines.csv', 'segment_dp_omegas_v2_mt-ae.csv',
          'rpmpq_v2_zeta.csv', 'rpmpq_v2_perfect_rates.csv',
          'mp_policy_lut_mamba_pruned.csv']:
    fp = os.path.join(csv_dir, f)
    status = 'OK' if os.path.exists(fp) else 'MISSING'
    print(f'  [{status}] {f}')

t_total = time.time()

# ── Step definitions ──
steps = [
    ('budget_allocation.csv',     'analysis/budget_allocation.py',           'CPU'),
    ('full_comparison.csv',       'analysis/eval_full_comparison.py',        'GPU'),
    ('outage_multi_snr_sweep.csv','analysis/eval_outage_multi_snr_sweep.py', 'GPU'),
    (None,                        'analysis/budget_allocation_outage.py',    'GPU'),  # multiple outputs
    (None,                        'analysis/generate_paper_figures.py',      'CPU'),  # figures
]
# Step 4 outputs
step4_outputs = ['outage_curves_per_bin.csv', 'budget_allocation_outage.csv', 'complete_eval_mtae.csv']

for i, (output_csv, script, hw) in enumerate(steps):
    step_num = i + 1

    # Skip if output already exists (not stale)
    if output_csv:
        fp = os.path.join(csv_dir, output_csv)
        if os.path.exists(fp):
            mtime = datetime.fromtimestamp(os.path.getmtime(fp))
            print(f'\n[SKIP] Step {step_num}/5: {output_csv} already exists ({mtime.strftime("%m/%d %H:%M")})')
            continue
    elif step_num == 4:
        # Check all step4 outputs
        all_exist = all(os.path.exists(os.path.join(csv_dir, f)) for f in step4_outputs)
        if all_exist:
            print(f'\n[SKIP] Step {step_num}/5: all outage CSVs already exist')
            continue
    elif step_num == 5:
        pass  # always run figure generation

    print(f'\n{"="*60}')
    print(f'  Step {step_num}/5: {os.path.basename(script)} ({hw})')
    print(f'{"="*60}')
    t0 = time.time()
    !python {script}
    print(f'  Done: {(time.time()-t0)/60:.1f} min')

elapsed_total = (time.time() - t_total) / 60
print(f'\n{"="*60}')
print(f'  ALL DONE — Total: {elapsed_total:.1f} min')
print(f'{"="*60}')

# ── Verify all outputs ──
import pandas as pd
print('\n=== Output Verification ===')
for f in ['budget_allocation.csv', 'full_comparison.csv',
          'outage_multi_snr_sweep.csv', 'outage_curves_per_bin.csv',
          'budget_allocation_outage.csv', 'complete_eval_mtae.csv']:
    fp = os.path.join(csv_dir, f)
    if os.path.exists(fp):
        df = pd.read_csv(fp)
        mtime = datetime.fromtimestamp(os.path.getmtime(fp))
        for col in ['saving', 'target_saving']:
            if col in df.columns:
                vals = sorted(df[col].unique())
                step = min(vals[i+1]-vals[i] for i in range(min(5,len(vals)-1))) if len(vals)>1 else 0
                break
        else:
            step = 0
        print(f'  [OK] {f}: {len(df)} rows, step={step:.2f}%, updated {mtime.strftime("%H:%M")}')
    else:
        print(f'  [FAIL] {f}: NOT FOUND')

# Sanity check
fc_path = os.path.join(csv_dir, 'full_comparison.csv')
if os.path.exists(fc_path):
    df = pd.read_csv(fc_path)
    for sav in [85, 90, 93]:
        static = df[(df['method']=='nmse-static') & (abs(df['saving']-sav)<0.15)]
        opt = df[(df['method']=='nmse-adaptive-opt') & (abs(df['saving']-sav)<0.15)]
        if len(static)>0 and len(opt)>0:
            gap = opt['nmse_db'].iloc[0] - static['nmse_db'].iloc[0]
            ok = 'OK' if abs(gap)<0.5 else 'WARN'
            print(f'  [{ok}] @{sav}%: Static={static["nmse_db"].iloc[0]:.2f}, '
                  f'JointDP={opt["nmse_db"].iloc[0]:.2f}, gap={gap:+.2f} dB')